In [ ]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 50.4 MB/s eta 0:00:00


In [ ]:
import fitz  # PyMuPDF
import re

def extract_and_clean_pdf(pdf_path, output_txt_path="the-hobbit.txt", start_page=0, end_page=None):
    """
    استخراج متن از PDF، تمیزکاری اولیه و ذخیره به عنوان فایل txt
    """
    doc = fitz.open(pdf_path)
    total_pages = len(doc)

    if end_page is None or end_page > total_pages:
        end_page = total_pages

    print(f"Loading PDF with {total_pages} pages. Extracting from page {start_page} to {end_page}...")

    extracted_pages = []
    for page_num in range(start_page, end_page):
        page = doc[page_num]
        text = page.get_text()
        extracted_pages.append(text)

    full_text = "\n".join(extracted_pages)

    # --- پاک‌سازی اولیه مناسب برای LLM ---
    # ۱. چسباندن کلماتی که در انتهای خط با خط‌تیره نصف شده‌اند (مثلاً jour-\nney -> journey)
    full_text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', full_text)

    # ۲. تبدیل تب‌ها به فاصله معمولی
    full_text = full_text.replace('\t', ' ')

    # ۳. حذف فاصله‌های تکراری پشت سر هم
    full_text = re.sub(r'[ ]+', ' ', full_text)

    # ۴. کنترل اینترهای بیش از حد (تبدیل به حداکثر دو خط خالی)
    full_text = re.sub(r'\n{3,}', '\n\n', full_text)

    # ذخیره در فایل txt با انکودینگ utf-8
    with open(output_txt_path, "w", encoding="utf-8") as f:
        f.write(full_text)

    print(f"Extraction finished! Saved to {output_txt_path}")
    print(f"Total characters: {len(full_text):,}")
    return full_text

In [1]:
from importlib.metadata import version
import torch

print("torch version:", version("torch"))


from previous_chapters import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
# model.out_head.weight = model.tok_emb.weight
model.eval();  # Disable dropout during inference

torch version: 2.11.0+cu128


In [2]:
def split_text_interleaved(text, num_blocks=10, train_ratio=0.9):
    """
    متن را به N بخش در طول خط زمانی کتاب تقسیم کرده
    و از هر بخش، ۹۰ درصد برای آموزش و ۱۰ درصد برای ارزیابی برمی‌دارد.
    """
    block_size = len(text) // num_blocks
    train_chunks = []
    val_chunks = []

    for i in range(num_blocks):
        start = i * block_size
        end = (i + 1) * block_size if i < num_blocks - 1 else len(text)
        block = text[start:end]

        # نقطه برش بر اساس نسبت تعیین‌شده
        split_point = int(len(block) * train_ratio)

        # پیدا کردن نزدیک‌ترین فاصله برای جلوگیری از قطع شدن کلمات در مرز برش
        split_point = block.rfind(" ", 0, split_point)
        if split_point == -1:
            split_point = int(len(block) * train_ratio)

        train_chunks.append(block[:split_point].strip())
        val_chunks.append(block[split_point:].strip())

    train_text = "\n\n".join(train_chunks)
    val_text = "\n\n".join(val_chunks)

    return train_text, val_text

In [3]:
import os
import torch
import tiktoken
from previous_chapters import GPTDatasetV1, create_dataloader_v1  # توابع فصول قبل کتاب
# from previous_chapters import GPTModel

pdf_file = "The Hobbit.pdf"
txt_file = "lotr-full.txt"
output_txt_path = "lotr-full.txt"
txt_file1 = "lotr1.txt"
txt_file2 = "lotr2.txt"
txt_file3 = "lotr3.txt"
pdf_file1 = "1.pdf"
pdf_file2 = "2.pdf"
pdf_file3 = "3.pdf"
# اگر قبلاً استخراج نشده، استخراج کن؛ وگرنه از txt بخوان
if not os.path.exists(txt_file):
    # به عنوان مثال: شروع از صفحه ۱۰ تا صفحه ۳۰۰ (بسته به نسخه PDF خودت تنظیم کن)
    text_data = extract_and_clean_pdf(pdf_file1, output_txt_path=txt_file1, start_page=22, end_page=None)
    text_data += extract_and_clean_pdf(pdf_file2, output_txt_path=txt_file2, start_page=6, end_page=245)
    text_data += extract_and_clean_pdf(pdf_file3, output_txt_path=txt_file3, start_page=11, end_page=302)

    with open(output_txt_path, "w", encoding="utf-8") as f:
        f.write(text_data)
else:
    with open(txt_file, "r", encoding="utf-8") as file:
        text_data = file.read()

# تقسیم Train و Validation
train_ratio = 0.8
# split_idx = int(train_ratio * len(text_data))
# تقسیم متعادل کتاب به ۱۰ بلوک زمانی
train_text, val_text = split_text_interleaved(text_data, num_blocks=10, train_ratio=0.9)

print(f"Train characters: {len(train_text):,} (~{len(train_text)//4:,} tokens)")
print(f"Val characters:   {len(val_text):,} (~{len(val_text)//4:,} tokens)")

torch.manual_seed(123)

# ساخت Dataloaderها دقیقا مثل کد کتاب
train_loader = create_dataloader_v1(
    train_text,
    batch_size=16,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=int(GPT_CONFIG_124M["context_length"]/4),
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_text,
    batch_size=16,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=int(GPT_CONFIG_124M["context_length"]/4),
    drop_last=False,
    shuffle=False,
    num_workers=0
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train characters: 2,232,063 (~558,015 tokens)
Val characters:   248,059 (~62,014 tokens)
Train batches: 595
Val batches: 66


In [6]:
n_epochs = 6
initial_lr = 0.0001
peak_lr = 0.01

In [7]:
total_steps = len(train_loader) * n_epochs
warmup_steps = int(0.2 * total_steps) # 20% warmup
print(warmup_steps)

1428


In [ ]:
from previous_chapters import evaluate_model, generate_and_print_sample, calc_loss_batch
import math

def train_model(model, train_loader, val_loader, optimizer, device,
                n_epochs, eval_freq, eval_iter, start_context, tokenizer,
                warmup_steps, initial_lr=3e-05, min_lr=1e-6):

    train_losses, val_losses, track_tokens_seen, track_lrs = [], [], [], []
    tokens_seen, global_step = 0, -1

    # Retrieve the maximum learning rate from the optimizer
    peak_lr = optimizer.param_groups[0]["lr"]

    # Calculate the total number of iterations in the training process
    total_training_steps = len(train_loader) * n_epochs

    # Calculate the learning rate increment during the warmup phase
    lr_increment = (peak_lr - initial_lr) / warmup_steps

    for epoch in range(n_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            global_step += 1

            # Adjust the learning rate based on the current phase (warmup or cosine annealing)
            if global_step < warmup_steps:
                # Linear warmup
                lr = initial_lr + global_step * lr_increment
            else:
                # Cosine annealing after warmup
                progress = ((global_step - warmup_steps) /
                            (total_training_steps - warmup_steps))
                lr = min_lr + (peak_lr - min_lr) * 0.5 * (1 + math.cos(math.pi * progress))

            # Apply the calculated learning rate to the optimizer
            for param_group in optimizer.param_groups:
                param_group["lr"] = lr
            track_lrs.append(lr)  # Store the current learning rate

            # Calculate and backpropagate the loss
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()

            # Apply gradient clipping after the warmup phase to avoid exploding gradients
            if global_step > warmup_steps:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            tokens_seen += input_batch.numel()

            # Periodically evaluate the model on the training and validation sets
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader,
                    device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                # Print the current losses
                print(f"Ep {epoch+1} (Iter {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, "
                      f"Val loss {val_loss:.3f}"
                )

        # Generate and print a sample from the model to monitor progress
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen, track_lrs

In [ ]:
import tiktoken

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.out_head.weight = model.tok_emb.weight
model.to(device)

peak_lr = 2e-4
optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0.2)
tokenizer = tiktoken.get_encoding("gpt2")

n_epochs = 6
train_losses, val_losses, tokens_seen, lrs = train_model(
    model, train_loader, val_loader, optimizer, device, n_epochs=n_epochs,
    eval_freq=5, eval_iter=1, start_context="When Bilbo came to himself",
    tokenizer=tokenizer, warmup_steps=warmup_steps,
    initial_lr=1e-5, min_lr=1e-5
)

Ep 1 (Iter 000000): Train loss 10.995, Val loss 10.963
Ep 1 (Iter 000005): Train loss 10.872, Val loss 10.865
Ep 1 (Iter 000010): Train loss 10.741, Val loss 10.724
Ep 1 (Iter 000015): Train loss 10.534, Val loss 10.534
Ep 1 (Iter 000020): Train loss 10.303, Val loss 10.288
Ep 1 (Iter 000025): Train loss 10.046, Val loss 10.011
Ep 1 (Iter 000030): Train loss 9.741, Val loss 9.751
Ep 1 (Iter 000035): Train loss 9.482, Val loss 9.519
Ep 1 (Iter 000040): Train loss 9.184, Val loss 9.311
Ep 1 (Iter 000045): Train loss 9.095, Val loss 9.116
Ep 1 (Iter 000050): Train loss 8.859, Val loss 8.920
Ep 1 (Iter 000055): Train loss 8.684, Val loss 8.717
Ep 1 (Iter 000060): Train loss 8.430, Val loss 8.513
Ep 1 (Iter 000065): Train loss 8.277, Val loss 8.310
Ep 1 (Iter 000070): Train loss 8.066, Val loss 8.107
Ep 1 (Iter 000075): Train loss 7.803, Val loss 7.914
Ep 1 (Iter 000080): Train loss 7.733, Val loss 7.719
Ep 1 (Iter 000085): Train loss 7.410, Val loss 7.532
Ep 1 (Iter 000090): Train loss 7.2

KeyboardInterrupt: 

In [4]:
import math
import torch
from previous_chapters import evaluate_model, generate_and_print_sample, calc_loss_batch

def train_model(
    model, train_loader, val_loader, optimizer, device,
    n_epochs, eval_freq, eval_iter, start_context, tokenizer,
    warmup_steps, initial_lr=3e-05, min_lr=1e-6,
    peak_lr=None, accumulation_steps=4, save_path="best_model.pt", resume_path=None
):
    train_losses, val_losses, track_tokens_seen, track_lrs = [], [], [], []
    tokens_seen = 0
    global_step = -1
    start_epoch = 0
    best_val_loss = float("inf")
    # ۱. بازیابی وضعیت در صورت درخواست Resume
    if resume_path and os.path.exists(resume_path):
        print(f"Loading checkpoint from: {resume_path}")
        checkpoint = torch.load(resume_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = checkpoint["epoch"] + 1
        global_step = checkpoint["global_step"]
        best_val_loss = checkpoint.get("best_val_loss", float("inf"))
        tokens_seen = checkpoint.get("tokens_seen", 0)
        print(f"Resumed from Epoch {start_epoch}, Step {global_step}, Best Val Loss: {best_val_loss:.3f}")


    # ۱. تنظیم مطمئن peak_lr
    if peak_lr is None:
        peak_lr = optimizer.param_groups[0]["lr"]
    else:
        for param_group in optimizer.param_groups:
            param_group["lr"] = initial_lr

    # ۲. محاسبه گام‌های واقعی بهینه‌سازی (بر اساس تعداد دفعات آپدیت وزن)
    steps_per_epoch = math.ceil(len(train_loader) / accumulation_steps)
    total_training_steps = steps_per_epoch * n_epochs

    lr_increment = (peak_lr - initial_lr) / warmup_steps

    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()  # صفر کردن گرادیان در آغاز هر اپوک

        for batch_idx, (input_batch, target_batch) in enumerate(train_loader):
            # محاسبه لاس و تقسیم بر تعداد مراحل تجمع
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss = loss / accumulation_steps
            loss.backward()

            tokens_seen += input_batch.numel()

            # بررسی رسیدن به زمان آپدیت وزن‌ها (یا آخرین بچ اپوک)
            is_accumulation_done = ((batch_idx + 1) % accumulation_steps == 0)
            is_last_batch = (batch_idx + 1 == len(train_loader))

            if is_accumulation_done or is_last_batch:
                global_step += 1

                # اعمال اسکژولر (Warmup + Cosine Decay)
                if global_step < warmup_steps:
                    lr = initial_lr + global_step * lr_increment
                else:
                    progress = (global_step - warmup_steps) / max(1, (total_training_steps - warmup_steps))
                    lr = min_lr + (peak_lr - min_lr) * 0.5 * (1 + math.cos(math.pi * progress))

                for param_group in optimizer.param_groups:
                    param_group["lr"] = lr
                track_lrs.append(lr)

                # کلیپ کردن گرادیان‌ها بعد از اتمام وارم‌آپ و قبل از آپدیت
                if global_step > warmup_steps:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                # آپدیت وزن‌ها و پاک‌سازی گرادیان‌ها برای چرخه بعدی
                optimizer.step()
                optimizer.zero_grad()

                # ارزیابی دوره‌ای
                if global_step % eval_freq == 0:
                    train_loss, val_loss = evaluate_model(
                        model, train_loader, val_loader,
                        device, eval_iter
                    )
                    train_losses.append(train_loss)
                    val_losses.append(val_loss)
                    track_tokens_seen.append(tokens_seen)

                    # لاگین بهبود و ذخیره چک‌پوینت
                    saved_indicator = ""
                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        checkpoint_data = {
                            "epoch": epoch,
                            "global_step": global_step,
                            "model_state_dict": model.state_dict(),
                            "optimizer_state_dict": optimizer.state_dict(),
                            "best_val_loss": best_val_loss,
                            "tokens_seen": tokens_seen
                        }
                        torch.save(checkpoint_data, save_path)
                        saved_indicator = " -> [SAVED NEW BEST]"

                    print(
                        f"Ep {epoch+1} (Step {global_step:06d}): "
                        f"Train loss {train_loss:.3f}, "
                        f"Val loss {val_loss:.3f}, "
                        f"Difference : {(val_loss - train_loss):.3f},"
                        f"LR {lr:.2e}"
                    )

        latest_data = {
                "epoch": epoch,
                "global_step": global_step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "tokens_seen": tokens_seen
            }
        torch.save(latest_data, "last_checkpoint.pt")
        # تولید متن نمونه در پایان هر اپوک
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen, track_lrs

In [5]:
import tiktoken
# فعال‌سازی اشتراک وزن برای کاهش پارامترهای هدر رفته دیکشنری
def init_weights(module):
    if isinstance(module, (torch.nn.Linear, torch.nn.Embedding)):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, torch.nn.Linear) and module.bias is not None:
            torch.nn.init.zeros_(module.bias)

model = GPTModel(GPT_CONFIG_124M)
# model.apply(init_weights)
# model.out_head.weight = model.tok_emb.weight
model.to(device)

peak_lr = 3e-4  # نرخ یادگیری کنترل‌شده

# مقداردهی صریح lr به AdamW
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=peak_lr,
    weight_decay=0.05
)
tokenizer = tiktoken.get_encoding("gpt2")

# با batch_size=12 و accumulation_steps=4، بچ موثر برابر ۴۸ است (۱۲ × ۴)
train_losses, val_losses, tokens_seen, lrs = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    n_epochs=8,
    eval_freq=15,
    eval_iter=5,
    start_context="When Bilbo came to himself",
    tokenizer=tokenizer,
    warmup_steps=80,
    initial_lr=1e-5,
    min_lr=1e-5,
    peak_lr=peak_lr,
    accumulation_steps=4,
    save_path="best_model.pt",
    resume_path=None
)

Ep 1 (Step 000000): Train loss 10.901, Val loss 10.910, Difference : 0.009,LR 1.00e-05
Ep 1 (Step 000015): Train loss 8.878, Val loss 8.975, Difference : 0.097,LR 6.44e-05
Ep 1 (Step 000030): Train loss 7.470, Val loss 7.660, Difference : 0.190,LR 1.19e-04
Ep 1 (Step 000045): Train loss 6.418, Val loss 6.667, Difference : 0.249,LR 1.73e-04
Ep 1 (Step 000060): Train loss 6.001, Val loss 6.312, Difference : 0.311,LR 2.27e-04
Ep 1 (Step 000075): Train loss 5.628, Val loss 5.999, Difference : 0.371,LR 2.82e-04
Ep 1 (Step 000090): Train loss 5.338, Val loss 5.749, Difference : 0.411,LR 3.00e-04
Ep 1 (Step 000105): Train loss 5.199, Val loss 5.591, Difference : 0.392,LR 3.00e-04
Ep 1 (Step 000120): Train loss 4.976, Val loss 5.462, Difference : 0.486,LR 2.99e-04
Ep 1 (Step 000135): Train loss 4.876, Val loss 5.385, Difference : 0.508,LR 2.98e-04
When Bilbo came to himself. ‘I have you,’ said Frodo. ‘I am not know,’ said Frodo. ‘I have ’ said Pippin. ‘I am not ’ said Frodo
Ep 2 (Step 000150):

KeyboardInterrupt: 

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import shutil

shutil.move('best_model.pt', '/content/drive/MyDrive/best_model-aproach2.pt')

FileNotFoundError: [Errno 2] No such file or directory: 'best_model.pt'

In [9]:
from previous_chapters import plot_losses
import matplotlib.pyplot as plt
n_epochs = 5
epochs_tensor = torch.linspace(1, n_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)
plt.tight_layout(); plt.savefig("3.pdf")
plt.show()

NameError: name 'train_losses' is not defined

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    },
    "/content/drive/MyDrive/train_lotr/model_and_optimizer.pth"
)

In [ ]:
checkpoint = torch.load("model_and_optimizer.pth")

model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();